# Use the best AutoML generated model to batch score credit worthiness

<img src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-ml-experiment.png" style="float: right" width="600px">


Databricks AutoML runs experiments across a grid and creates many models and metrics to determine the best models among all trials. This is a glass-box approach to create a baseline model, meaning we have all the code artifacts and experiments available afterwards. 

Here, we selected the Notebook from the best run from the AutoML experiment.

All the code below has been automatically generated. As data scientists, we can tune it based on our business knowledge, or use the generated model as-is.

This saves data scientists hours of developement and allows team to quickly bootstrap and validate new projects, especally when we may not know the predictors for alternative data such as the telco payment data.

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F03-Data-Science-ML%2F03.3-Batch-Scoring-credit-decisioning&demo_name=lakehouse-fsi-credit&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-fsi-credit%2F03-Data-Science-ML%2F03.3-Batch-Scoring-credit-decisioning&version=1">

In [0]:
%pip install mlflow==2.19.0
dbutils.library.restartPython()

  Using cached mlflow-2.19.0-py3-none-any.whl.metadata (30 kB)
  Using cached mlflow_skinny-2.19.0-py3-none-any.whl.metadata (31 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
Using cached mlflow-2.19.0-py3-none-any.whl (27.4 MB)
Using cached mlflow_skinny-2.19.0-py3-none-any.whl (5.9 MB)
Using cached docker-7.1.0-py3-none-any.whl (147 kB)
Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
Using cached graphql_relay-3.2.0-py3-none-any.whl (16 kB)
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 2.21.3
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4fdd613d-92e5-46aa-9f95-d194127a8499
    Can't uninstall 'mlflow-skinny'. No files were found to uninstall.
Note: you may need to restart the

In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_fsi_credit_decisioning`



## Running batch inference to score our existing database

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/credit_decisioning/fsi-credit-decisioning-ml-5.png" style="float: right" width="800px">

<br/><br/>
Now that our model was created and deployed in production within the MLFlow registry.

<br/>
We can now easily load it calling the `Production` stage, and use it in any Data Engineering pipeline (a job running every night, in streaming or even within a Spark Declarative Pipelines pipeline).

<br/>

We'll then save this information as a new table without our FS database, and start building dashboards and alerts on top of it to run live analysis.

In [0]:
model_name = "dbdemos_fsi_credit_decisioning"
import mlflow
mlflow.set_registry_uri('databricks-uc')

# Load model as a Spark UDF.
loaded_model = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{catalog}.{db}.{model_name}@prod", result_type='double')

2025/12/04 20:06:33 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 2.19.0, required: mlflow==2.21.3)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/12/04 20:06:33 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/12/04 20:06:34 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [0]:
features = loaded_model.metadata.get_input_schema().input_names()

underbanked_df = spark.table("credit_decisioning_features").fillna(0) \
                   .withColumn("prediction", loaded_model(F.struct(*features))).cache()

display(underbanked_df)

cust_id,education,marital_status,months_current_address,months_employment,is_resident,tenure_months,product_cnt,tot_rel_bal,revenue_tot,revenue_12m,income_annual,tot_assets,overdraft_balance_amount,overdraft_number,total_deposits_number,total_deposits_amount,total_equity_amount,total_UT,customer_revenue,age,avg_balance,num_accs,balance_usd,available_balance_usd,is_pre_paid,number_payment_delays_last12mo,pct_increase_annual_number_of_delays_last_3_year,phone_bill_amt,avg_phone_bill_amt_lst12mo,dist_payer_cnt_12m,sent_txn_cnt_12m,sent_txn_amt_12m,sent_amt_avg_12m,dist_payee_cnt_12m,rcvd_txn_cnt_12m,rcvd_txn_amt_12m,rcvd_amt_avg_12m,dist_payer_cnt_6m,sent_txn_cnt_6m,sent_txn_amt_6m,sent_amt_avg_6m,dist_payee_cnt_6m,rcvd_txn_cnt_6m,rcvd_txn_amt_6m,rcvd_amt_avg_6m,dist_payer_cnt_3m,sent_txn_cnt_3m,sent_txn_amt_3m,sent_amt_avg_3m,dist_payee_cnt_3m,rcvd_txn_cnt_3m,rcvd_txn_amt_3m,rcvd_amt_avg_3m,tot_txn_cnt_12m,tot_txn_amt_12m,tot_txn_cnt_6m,tot_txn_amt_6m,tot_txn_cnt_3m,tot_txn_amt_3m,ratio_txn_amt_3m_12m,ratio_txn_amt_6m_12m,prediction
50,2,1,44,84,0,97,1,1724.38,488.28,0.42,102732,1027,9.174683159E7,0,1,4444568.95,4694267.39,7073893.86,7136809.0,30,221.0,1,221.0,1527.37,1,0,0,41.98,41.98,0,0,0.0,0.0,1,3,652.39,217.46333333333334,0,0,0.0,0.0,1,1,273.64,273.64,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,1.0
518,3,0,63,1,1,149,2,19001.77,998.81,0.56,79932,79932,5.7386063884E8,7,8,6040180.0,1406166.33,4776694.0,2510863.0,26,2853.49,1,2853.49,2158.52,0,0,0,49.99,49.99,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,1.0
750,2,2,39,13,1,106,1,887.05,1185.08,0.93,113064,1130,6.6827429023E8,0,2,3650389.75,1358955.35,5125404.0,4351280.78,48,6195.85,1,6195.85,383.93,0,0,0,89.99,89.99,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,1.0
771,0,1,14,1,0,25,4,36.0,602.69,2.01,35124,0,5.4086133119E8,6,10,9841140.0,8151069.17,6874894.31,2929922.0,61,4113.1,1,4113.1,239.02,0,0,0,19.99,19.99,0,0,0.0,0.0,1,13,2727.92,209.84,0,0,0.0,0.0,1,6,1367.2800000000002,227.88000000000002,0,0,0.0,0.0,1,4,941.4200000000001,235.35500000000002,0,0.0,0,0.0,0,0.0,0.0,0.0,0.0
951,4,0,3,96,0,53,1,5871.96,932.9,1.47,90396,903,6.0880169558E8,4,7,1831504.53,5062367.0,8619780.0,6764816.0,34,1457.285,2,2623.82,1163.27,0,0,0,49.99,49.99,0,0,0.0,0.0,1,30,6022.539999999999,200.7513333333333,0,0,0.0,0.0,1,17,3462.6299999999997,203.6841176470588,0,0,0.0,0.0,1,9,2013.5700000000002,223.73000000000002,0,0.0,0,0.0,0,0.0,0.0,0.0,1.0
1214,0,0,1,56,1,112,2,18646.11,1268.21,0.94,55752,557,7.358438545E8,9,10,9097153.0,3723241.0,1394711.4,7189156.0,30,3864.9,1,3864.9,1282.52,0,0,0,19.99,19.99,0,0,0.0,0.0,1,7,1508.99,215.57,0,0,0.0,0.0,1,2,511.71000000000004,255.85500000000002,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,1.0
1414,4,2,56,73,0,1,4,8151.14,898.48,74.87,42720,0,5.6594330063E8,6,4,6437599.0,5272965.0,4135071.33,9910672.0,26,224.33333333333334,3,673.0,0.0,1,3,1,131.98,131.98,1,1,113.37,113.37,1,1,232.71,232.71,1,1,113.37,113.37,1,1,232.71,232.71,0,0,0.0,0.0,1,1,232.71,232.71,2,346.08000000000004,2,346.08000000000004,0,0.0,0.0,1.0,1.0
1660,4,2,27,25,0,22,1,16696.25,1094.99,4.15,33528,0,3.9601033965E8,7,6,4441911.0,1979218.44,495709.81,3384286.0,40,5305.79,1,5305.79,1140.73,1,0,0,56.98,56.98,0,0,0.0,0.0,1,9,1881.6699999999998,209.07444444444442,0,0,0.0,0.0,1,4,852.96,213.24,0,0,0.0,0.0,1,3,630.12,210.04,0,0.0,0,0.0,0,0.0,0.0,0.0,1.0
2395,3,1,42,54,1,43,1,5979.89,1713.73,3.32,113016,113016,5.0228380795E8,5,5,2128987.22,3899320.27,3542372.0,841168.53,49,4859.1,1,4859.1,2984.05,0,0,0,124.99,124.99,0,0,0.0,0.0,1,4,777.24,194.31,0,0,0.0,0.0,1,4,777.24,194.31,0,0,0.0,0.0,1,1,216.32,216.32,0,0.0,0,0.0,0,0.0,0.0,0.0,0.0
2431,3,2,83,41,1,38,3,23109.45,155.88,0.34,102072,1020,9.827944674E8,9,2,5354286.48,6501036.89,3717010.0,1147754.17,31,4258.545,2,8112.09,1620.15,0,0,0,19.99,19.99,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0,1.0


In the scored data frame above, we have essentially created an end-to-end process to predict credit worthiness for any customer, regardless of whether the customer has an existing bank account. We have a binary prediction which captures this and incorporates all the intellience from Databricks AutoML and curated features from our feature store.

In [0]:
underbanked_df.write.mode("overwrite").saveAsTable(f"underbanked_prediction")


### Next steps

* Deploy your model for real time inference with [03.4-model-serving-BNPL-credit-decisioning]($./03.4-model-serving-BNPL-credit-decisioning) to enable ```Buy Now, Pay Later``` capabilities within the bank.

Or

* Making sure your model is fair towards customers of any demographics are extremely important parts of building production-ready ML models for FSI use cases. <br/>
Explore your model with [03.5-Explainability-and-Fairness-credit-decisioning]($./03.5-Explainability-and-Fairness-credit-decisioning) on the Lakehouse.